# MC Sim - GPU Build & Test

Build and test the molecular communication GPU simulator on Colab.

**Before running:**
1. Go to Runtime > Change runtime type > T4 GPU
2. Click the 🔑 key icon in the left sidebar, add a secret named `GITHUB_PAT` with your token, and toggle notebook access on

In [ ]:
# Verify GPU is available
!nvidia-smi
!nvcc --version

In [ ]:
# Clone the repo (uses GITHUB_PAT from Colab secrets)
from google.colab import userdata
PAT = userdata.get('GITHUB_PAT')
!git clone https://{PAT}@github.com/alwaysEpic/molecular_modeling_gpu.git
%cd molecular_modeling_gpu

In [ ]:
# Build both targets + RNG dump tool
!mkdir -p build && cd build && cmake .. && make -j$(nproc)
!g++ -O2 -o build/dump_rng_cpu scripts/dump_rng_cpu.cpp -lm
!pip install -q numpy matplotlib scipy
!ls -la build/mc_sim build/mc_sim_cpu build/dump_rng_cpu

## 1. GPU Smoke Test

In [ ]:
!cd build && ./mc_sim -i 1000 -f -v

## 2. CPU vs GPU Comparison

In [ ]:
# 1000 paths
!cd build && ./mc_sim -i 1000 -c -f -v

In [ ]:
# 10000 paths
!cd build && ./mc_sim -i 10000 -c -f -v

## 3. CPU-Only Benchmarks

In [ ]:
!cd build && ./mc_sim_cpu -i 1000 -f -v
print("---")
!cd build && ./mc_sim_cpu -i 1000 -f -v -n

## 4. Analytical Validation — 1D First-Hit with Drift

Compares simulation output against analytical inverse Gaussian (thesis eq 4.3).

In [ ]:
# GPU validation
!cd build && ./mc_sim -i 10000 -f -l 3E-7 -t 1E-2 -v
print()
!python scripts/validate_1d_firsthit.py build/output_gpu.csv \
    --dist 3E-7 --vel 1E-4 --timestep 1E-7 --timestop 1E-2

In [ ]:
# CPU validation (1k paths — quick check, not benchmark)
!cd build && ./mc_sim_cpu -i 1000 -f -l 3E-7 -t 1E-2 -v
print()
!python scripts/validate_1d_firsthit.py build/output_h.csv \
    --dist 3E-7 --vel 1E-4 --timestep 1E-7 --timestop 1E-2

## 5. Analytical Validation — 3D Spherical Receiver (no drift)

Compares against analytical H_Diff (thesis eq 4.1).

In [ ]:
# GPU - 3D diffusion only
!cd build && ./mc_sim -i 10000 -f -n -v
print()
!python scripts/validate_3d_diffusion.py build/output_gpu.csv \
    --no-plot --total-paths 10000

In [ ]:
# CPU - 3D diffusion only (1k paths — quick check)
!cd build && ./mc_sim_cpu -i 1000 -f -n -v
print()
!python scripts/validate_3d_diffusion.py build/output_h.csv \
    --no-plot --total-paths 1000

## 6. CPU/GPU Agreement (Deep and Wide)

KS test comparing hit-time distributions from CPU against both GPU kernel paths.

In [ ]:
# CPU vs GPU Deep agreement
print("=== CPU vs GPU Deep ===")
!cd build && ./mc_sim -i 5000 -c -f -l 3E-7 -t 1E-2 > /dev/null 2>&1
!python scripts/validate_cpu_gpu_agreement.py build/output_h.csv build/output_gpu.csv \
    --timestep 1E-7 --no-plot
print()
# CPU vs GPU Wide agreement
print("=== CPU vs GPU Wide ===")
!cd build && ./mc_sim -i 5000 -c -f -l 3E-7 -t 1E-2 -W > /dev/null 2>&1
!python scripts/validate_cpu_gpu_agreement.py build/output_h.csv build/output_gpu.csv \
    --timestep 1E-7 --no-plot

## 7. RNG Quality

Tests statistical moments, autocorrelation, and normality.

In [ ]:
# CPU RNG quality
!cd build && ./dump_rng_cpu 10000 > rng_cpu.csv
!python scripts/validate_rng.py build/rng_cpu.csv --no-plot

In [ ]:
%%writefile /tmp/dump_rng_gpu.cu
// Dump GPU random numbers for RNG quality testing
#include <stdio.h>
#include <stdlib.h>
#include <curand.h>
#include <curand_kernel.h>

__global__ void gen_randn(curandStatePhilox4_32_10_t* rng, float* out, int n, int draws_per_thread) {
  int idx = blockIdx.x * blockDim.x + threadIdx.x;
  if (idx >= n) return;
  for (int d = 0; d < draws_per_thread; d++) {
    out[idx * draws_per_thread + d] = curand_normal(&rng[idx]);
  }
}

__global__ void init_rng(curandStatePhilox4_32_10_t* rng, long long seed, int n) {
  int idx = blockIdx.x * blockDim.x + threadIdx.x;
  if (idx >= n) return;
  curand_init(seed, idx, 0, &rng[idx]);
}

int main(int argc, char** argv) {
  int n_threads = 1000;
  int draws = 10;  // 10 draws per thread = 10000 total
  int total = n_threads * draws;
  long long seed = 42;
  if (argc > 1) seed = atoll(argv[1]);

  curandStatePhilox4_32_10_t* d_rng;
  float* d_out;
  cudaMalloc(&d_rng, n_threads * sizeof(curandStatePhilox4_32_10_t));
  cudaMalloc(&d_out, total * sizeof(float));

  int block = 128;
  int grid = (n_threads + block - 1) / block;
  init_rng<<<grid, block>>>(d_rng, seed, n_threads);
  cudaDeviceSynchronize();
  gen_randn<<<grid, block>>>(d_rng, d_out, n_threads, draws);
  cudaDeviceSynchronize();

  float* out = (float*)malloc(total * sizeof(float));
  cudaMemcpy(out, d_out, total * sizeof(float), cudaMemcpyDeviceToHost);

  for (int i = 0; i < total; i++) printf("%0.15f\n", out[i]);

  free(out);
  cudaFree(d_rng);
  cudaFree(d_out);
  return 0;
}

In [ ]:
# Compile and run GPU RNG dump, then validate
!nvcc -O2 -o build/dump_rng_gpu /tmp/dump_rng_gpu.cu
!cd build && ./dump_rng_gpu 42 > rng_gpu.csv
print("=== GPU RNG (persistent Philox, seed=42) ===")
!python scripts/validate_rng.py build/rng_gpu.csv --no-plot
print()
!cd build && ./dump_rng_gpu 123 > rng_gpu2.csv
print("=== GPU RNG (persistent Philox, seed=123) ===")
!python scripts/validate_rng.py build/rng_gpu2.csv --no-plot

## 8. Reproducibility (--seed)

Same seed should produce identical output.

In [ ]:
# GPU reproducibility
!cd build && ./mc_sim -i 500 -f -l 3E-7 -t 1E-3 -S 42 && mv output_gpu.csv run1_gpu.csv
!cd build && ./mc_sim -i 500 -f -l 3E-7 -t 1E-3 -S 42 && mv output_gpu.csv run2_gpu.csv
!diff build/run1_gpu.csv build/run2_gpu.csv && echo 'GPU REPRODUCIBILITY: PASS (identical)' || echo 'GPU REPRODUCIBILITY: FAIL (differs)'

In [ ]:
# CPU reproducibility
!cd build && ./mc_sim_cpu -i 500 -f -l 3E-7 -t 1E-3 -S 42 && mv output_h.csv run1_cpu.csv
!cd build && ./mc_sim_cpu -i 500 -f -l 3E-7 -t 1E-3 -S 42 && mv output_h.csv run2_cpu.csv
!diff build/run1_cpu.csv build/run2_cpu.csv && echo 'CPU REPRODUCIBILITY: PASS (identical)' || echo 'CPU REPRODUCIBILITY: FAIL (differs)'

## 9. Performance Regression Test

Builds the current branch and thesis-baseline branch, runs both back-to-back
in the same session, and fails if current is measurably slower.

In [ ]:
# Build thesis-baseline for comparison
import os, subprocess
from google.colab import userdata

if not os.path.isdir('/content/baseline_build'):
    PAT = userdata.get('GITHUB_PAT')
    !git clone https://{PAT}@github.com/alwaysEpic/molecular_modeling_gpu.git /content/baseline_build
    !cd /content/baseline_build && git checkout thesis-baseline
    !cd /content/baseline_build && mkdir -p build && cd build && cmake .. 2>&1 | tail -1 && make -j$(nproc) 2>&1 | tail -1
else:
    print("Baseline already built")

In [ ]:
import subprocess, re, os

def run_and_time(binary_path, args, cwd, runs=3):
    """Run binary multiple times, extract GPU time, return median."""
    times = []
    for _ in range(runs):
        result = subprocess.run(
            [binary_path] + args,
            capture_output=True, text=True, cwd=cwd
        )
        output = result.stdout + result.stderr
        match = re.search(r'GPU code execution time is ([\d.]+)s', output)
        if match:
            times.append(float(match.group(1)))
    return sorted(times)[len(times)//2] if times else None

# Paths
current_dir = os.getcwd()
current_bin = os.path.join(current_dir, "build", "mc_sim")
baseline_bin = "/content/baseline_build/build/mc_sim"

tests = [
    ("1k default first-hit", ["-i", "1000", "-f", "-v"]),
    ("10k default first-hit", ["-i", "10000", "-f", "-v"]),
    ("10k 1D limit", ["-i", "10000", "-f", "-l", "3E-7", "-t", "1E-2", "-v"]),
]

print("=== Performance Regression Test (median of 3 runs) ===\n")
print(f"{'Test':<25} {'Baseline':>10} {'Current':>10} {'Change':>10} {'Status':>8}")
print("-" * 70)

all_pass = True
for name, args in tests:
    t_base = run_and_time(baseline_bin, args, "/content/baseline_build/build")
    t_curr = run_and_time(current_bin, args, os.path.join(current_dir, "build"))

    if t_base and t_curr:
        change = (t_curr - t_base) / t_base * 100
        # FAIL if current is >20% slower (allows for Colab noise)
        status = "FAIL" if change > 20 else "PASS"
        if status == "FAIL":
            all_pass = False
        print(f"{name:<25} {t_base:>9.3f}s {t_curr:>9.3f}s {change:>+9.1f}% {status:>8}")
    else:
        print(f"{name:<25} {'ERR':>10} {'ERR':>10} {'—':>10} {'SKIP':>8}")

print()
if all_pass:
    print("PASS — no performance regression detected")
else:
    print("FAIL — current branch is significantly slower than baseline")

## 10. Wall Reflection Validation

Ported from TobyThesisTest_walls.m. Transmitter and receiver positioned near
the vessel wall. Validates that wall reflection does not corrupt diffusion
statistics by comparing against free-space analytical solution.

In [ ]:
# Wall test: transmitter at (0, 7.8um, 0), receiver at (0, 7.8um, 50nm)
# Vessel radius 8um, no drift, 10k paths, 0.4ms duration
!cd build && ./mc_sim -i 10000 -f -w -n \
    --start-y 7.8E-6 \
    --rec-y 7.8E-6 --rec-z 50E-9 \
    -r 8E-6 -t 0.4E-3 -v
print()
!python scripts/validate_3d_walls.py build/output_gpu.csv \
    --total-paths 10000 --no-plot

## 10b. Wide Kernel Validation

Tests the per-step (wide) kernel path with --wide flag.
This is the architecture for future particle interactions.

In [ ]:
# Wide kernel: 1D limit validation (should match deep kernel results)
!cd build && ./mc_sim -i 10000 -f -l 3E-7 -t 1E-2 -W -v
print()
!python scripts/validate_1d_firsthit.py build/output_gpu.csv \
    --dist 3E-7 --vel 1E-4 --timestep 1E-7 --timestop 1E-2 --no-plot
print()
# Wide kernel: 3D spherical validation
!cd build && ./mc_sim -i 10000 -f -n -W -v
print()
!python scripts/validate_3d_diffusion.py build/output_gpu.csv \
    --no-plot --total-paths 10000

## 11. Stress Test (1M paths)

High-sample-count validation with maximum statistical power.
KS critical value at 1M paths (alpha=0.001) is ~0.002 — detects
any CDF discrepancy above 0.2%.

In [ ]:
# 1M paths, 1D limit with drift — ultimate KS test
!cd build && ./mc_sim -i 1000000 -f -l 3E-7 -t 1E-2 -v
print()
!python scripts/validate_1d_firsthit.py build/output_gpu.csv \
    --dist 3E-7 --vel 1E-4 --timestep 1E-7 --timestop 1E-2

In [ ]:
# 1M paths, 3D spherical receiver — ultimate binomial + KS test
!cd build && ./mc_sim -i 1000000 -f -n -v
print()
!python scripts/validate_3d_diffusion.py build/output_gpu.csv \
    --no-plot --total-paths 1000000

In [ ]:
# 100k paths, wall reflection stress test
!cd build && ./mc_sim -i 100000 -f -w -n \
    --start-y 7.8E-6 \
    --rec-y 7.8E-6 --rec-z 50E-9 \
    -r 8E-6 -t 0.4E-3 -v
print()
!python scripts/validate_3d_walls.py build/output_gpu.csv \
    --total-paths 100000 --no-plot

## 12. Maximum Scale Test (10M paths)

The thesis maximum was 200k paths. With the persistent kernel we can
do 50x more. 10M paths gives KS critical value ~0.0006 at alpha=0.001.

In [ ]:
# 10M paths, 1D limit — 50x beyond thesis maximum
import time
print("Starting 10M path simulation...")
t0 = time.time()
!cd build && ./mc_sim -i 10000000 -f -l 3E-7 -t 1E-2 -v
t1 = time.time()
print(f"\nTotal wall time: {t1-t0:.1f}s")
print()
!python scripts/validate_1d_firsthit.py build/output_gpu.csv \
    --dist 3E-7 --vel 1E-4 --timestep 1E-7 --timestop 1E-2 --no-plot